## Interested in viewing the bathymetry or mesh that was used in the model to create CORA? This notebook allows users to create a rasterized plot of the topobathy at the CORA-GEC model nodes and overlay the model mesh.

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Rudiger, P. (2023). Bay Trimesh. https://github.com/holoviz/datashader/blob/f23de596f9adcb8188d48e6b163c36c913cd9912/examples/user_guide/6_Trimesh.ipynb#L11***

_ _ 

### Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask
import intake
import xarray as xr
import holoviews as hv
import datashader as ds
import datashader.transfer_functions as tf
import datashader.utils as du
import geoviews as gv
from holoviews import opts
import holoviews.operation.datashader as dshade
from holoviews.operation.datashader import datashade, shade, rasterize
import cmocean
import zarr
hv.extension('bokeh')

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-GEC water levels (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_gec/native_grid/water_levels/)). ###
#### This dataset contains hourly water levels from 1979-2022 in the GEC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **map** chunking here, though it does not matter for this application.

In [ ]:
ds = catalog["CORA-GEC-fort.63-timeseries"].to_dask()
ds

In [ ]:
v = np.vstack((ds.x,ds.y, ds.depth)).T
verts = pd.DataFrame(v, columns=['x','y','z'])
points = gv.operation.project_points(gv.Points(verts, vdims=['z']))
tris = pd.DataFrame(ds['element'].values.astype('int')-1, columns=['v0','v1','v2'])

In [ ]:
opts.defaults(
    opts.Image(width=1000, height=700),
    opts.RGB(width=1000, height=700))
wireframe = datashade(hv.TriMesh((tris,verts)).edgepaths)
trimesh = hv.TriMesh((tris,hv.Points(verts, vdims='z')))
dmesh = dshade.rasterize(trimesh).opts(cmap=cmocean.cm.deep, colorbar=True, clabel='Depth (MSL)')
dmesh * wireframe